<h1>ASSIGNMENT 2 INSTRUCTIONS</h1>

**An agenticAI workflow is a process where an LLM-based app executes multiple steps to complete a task.**

**This assignment accomplishes the following objectives:**
1. Learn two key strategies to improve the quality and success rate of agenticAI workflows: tool use and reflection.
2. See how we can invoke tools to provide the LLM access to custom and built-in tools.
3.  Create an agenticAI workflow with reflection that prompts the LLM to reflect on (and critique) past output in order to improve it.

**In order to complete the assignmet:**
1. Section 1: Assignment setup. Just read through to understand what's already available.
2. Section 2: Learn how tools work. Complete coding steps 2.1, 2.2, 2.3, and 2.4.
3. Section 3:  Learn how reflection works. Complete coding steps 3.1, 3.2, 3.3, 3.4, and 3.5.

**To submit:**
1. Rename the assignment as Assignment2-\<asurite\>
2. Runtime -> Restart session and run all
3. Ctrl-s
4. File -> Download .ipynb
5. Submit on Canvas

<h1>SECTION 1: ASSIGNMENT SETUP (JUST READ THROUGH) </h1>


<h2>We'll first import the required libraries.</h2>




In [1]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# install the orchestration layer (aisuite) plus the provider SDKs (OpenAI, Groq)
# in Colab, this is typically needed once per new runtime
!pip install aisuite openai groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.4/84.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: docstring-parser
    Found existing installation: docstring_parser 0.17.0
    Uninstalling docstring_parser-0.17.0:
      Successfully uninstalled docstring_parser-0.17.0
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.60.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.2 which is incompatible.


<h2>We'll now setup the API KEYS and Clients.</h2>

In [2]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

from google.colab import userdata
import os
import aisuite
from openai import OpenAI

# read the API_KEYs from Colab Secrets and expose it as an env variable
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# create the client
aisuite_client = aisuite.Client()
openai_client = OpenAI()

# select the models you want to use
OPENAI_MODEL= "gpt-4.1-mini"
AISUITE_OPENAI_MODEL = "openai:"+OPENAI_MODEL
GROQ_MODEL = "llama-3.3-70b-versatile"
AISUITE_GROQ_MODEL = "groq:"+GROQ_MODEL

<h2>The reponses we get back from LLMs aren't always well-formatted. We'll use this utility function to print LLM responses.</h2>

In [3]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# a nifty utility function to cleanly diplay your chat outputs
from IPython.display import display, HTML, Markdown

def show(title, text):
    display(HTML(f"""
     <h4>{title}</h4>
     <div style="white-space: pre-wrap; padding-left: 24px;">{text}</div>"""))

<h2>Recall from Assignment 1 that the LLM was unable to report the current time. To address this limitation, we will define a custom tool that the LLM can call when needed to retrieve the current time.</h2>

In [4]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

from datetime import datetime

# use custom get_current_time tool to get current time
def get_current_time():
  """
    Use custom code to find current local date/time and day of week
    and return it.
  """
  now = datetime.now()
  return {
      "date_time": now.strftime("%Y-%m-%d %H:%M:%S"),
      "day_of_week": now.strftime("%A"),
      }

<h2>Also recall from Assignment 1 that an LLM only has direct access to the information it was trained on and is unaware of events that occurred after its knowledge cutoff date. To address this limitation, we will provide access to a web_search tool that the LLM can call when it needs to retrieve current information. Since the OpenAI API provides a built-in web search capability, we will use it directly rather than implementing a custom solution. </h2>

In [5]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

import json

# use OpenAI's built-in web_search tool to perform web_search
def openai_web_search(query: str):
  """
    Perform a web search using OpenAI's built-in web_search tool
    and return raw results.
  """
  resp = openai_client.responses.create(
      model=OPENAI_MODEL,
      input=query,
      tools=[{"type": "web_search"}],
  )
  return json.loads(resp.model_dump_json())

<h2>The run_openai_query function is similar to what we used in Assignment 1, with two key changes. First, it can now accept a list of tools and invoke them as needed. Second, instead of printing the LMM response and usage statistics inline, it returns both the model's response and relevant query details (including usage statistics and any tools that were called).</h2>

In [6]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# used for wall-clock timing for request
import time

def run_openai_query(system_prompt, user_prompt, tools=[]):
  """
    Execute the query, print the prompt, the response, and detailed usage stats.
    Supports tools.
  """

  # start the time before the API call
  start_time = time.time()

  # make chat-completions request thro aisuite, routed to OpenAI-backed model
  completion = aisuite_client.chat.completions.create(
      model=AISUITE_OPENAI_MODEL, # OpenAI-backed model
      messages=[
          {"role": "system", "content": system_prompt}, # system sets global behavior/instructions
           {"role": "user", "content": user_prompt} # user contains the actual task/prompt
      ],
      tools=tools,
      max_turns=3,
  )

  # compute end-to-end wall-clock time for API request
  elapsed_time = time.time() - start_time

  # response
  response = completion.choices[0].message.content.strip()

  # usage stats
  query_details = {}
  query_details["prompt_tokens"]=completion.usage.prompt_tokens
  query_details["completion_tokens"]=completion.usage.completion_tokens
  query_details["total_tokens"]=completion.usage.total_tokens
  if hasattr(completion.usage, 'total_time'):
    query_details["total_time"]=f"{completion.usage.total_time:.2f}s"
  else:
    query_details["total_time"]=f"{elapsed_time:.2f}s"

  # tools invoked
  tools_invoked=[]
  messages = completion.choices[0].intermediate_messages
  for m in messages or []:
    if hasattr(m, "tool_calls") and m.tool_calls:
      for t in m.tool_calls:
        tools_invoked.append(t.function.name)
  query_details["tools_invoked"]=tools_invoked

  return response, query_details


<h1>SECTION 2: LEARN HOW TOOLS WORK (COMPLETE STEPS 2.1 through 2.4)</h1>

<h2>[Complete Coding STEP-2.1 and STEP-2.2]:<br>
You will now test LLM behavior with and without the custom get_current_time tool. Provide a system_prompt and a user_prompt as indicated below. Then execute the cell to see how that affects LLM behavior. Read inline comments in the cell for expected behavior. Ensure this is reflected in the ouput you see. </h2>

In [7]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

# STEP-2.1: Recall that the system prompt sets the global behavior/instructions for the LLM.
#           Enter a system prompt of choice.
#           You will enter your system prompt within the quotes below.
#           Refer to Assignment 1 for an illustration.
system_prompt = ("You are a helpful assistant. If you do not have access to real-time information, state that you do not know.")

# STEP-2.2: Recall that the user prompt contains the actual task/prompt for the LLM.
#           Enter a user prompt to ask for the current time.
#           You will enter your user prompt within the quotes below.
#           Refer to Assignment 1 for an illustration.
user_prompt = ("What is the current date, time, and day of the week?")

# Recall from Assignment 1 that the LLM did not have access to current time.
# And hence, it couldn't provide the current time.
response, query_details = run_openai_query(system_prompt, user_prompt, tools=[])
show("----- WITHOUT TOOLS -----", "")
show("Query", user_prompt)
show("Query Response", response)
show("Query Details", query_details)

# We'll now give the LLM access to the custom get_current_time tool.
# So now it can provide current time.
# Notice that get_current_time shows up in tools_invoked under "Query Details".
response, query_details = run_openai_query(system_prompt, user_prompt, tools=[get_current_time])
show("----- WITH TOOLS -----", "")
show("Query", user_prompt)
show("Query Response", response)
show("Query Details", query_details)

<h2>[Complete Coding STEP-2.3 and STEP-2.4]: <br>
You will now test LLM behavior with and without the built-in web_search tool. Provide a system_prompt and a user_prompt as indicated below. Then execute the cell to see how that affects LLM behavior. Read inline comments in the cell for expected behavior. Ensure this is reflected in the ouput you see. </h2>

In [8]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

# STEP-2.3: Recall that the system prompt sets the global behavior/instructions for the LLM.
#           Enter a system prompt of choice.
#           You will enter your system prompt within the quotes below.
#           Refer to Assignment 1 for an illustration.
system_prompt = ("You are a research assistant capable of using web search to find the latest news and corporate updates.")

# STEP-2.4: Recall that the user prompt contains the actual task/prompt for the LLM.
#           Enter a user prompt to ask about the Netflix Warner Bros acquisition.
#           You will enter your user prompt within the quotes below.
#           Refer to Assignment 1 for an illustration.
user_prompt = ("Can you give me the details regarding the Netflix acquisition of Warner Bros Discovery content or any recent merger news between them?")
# Recall from Assignment 1 that LLMs don't have information on events
# that occured after their knowledge cut off date (June 2024).
# And hence, it couldn't provide current information.
response, query_details = run_openai_query(system_prompt, user_prompt, tools=[])
show("----- WITHOUT TOOLS -----", "")
show("Query", user_prompt)
show("Query Response", response)
show("Query Details", query_details)

# Let's give the LLM access to the OpenAI's built-in openai_web_search tool.
# So now it can provide current information.
# Notice that openai_web_search shows up in tools_invoked under "Query Details".
# Also notice that it will only invoke the tools it needs to respond.
response, query_details = run_openai_query(system_prompt, user_prompt, tools=[get_current_time, openai_web_search])
show("----- WITH TOOLS -----", "")
show("Query", user_prompt)
show("Query Response", response)
show("Query Details", query_details)

<h1>SECTION 3: LEARN HOW REFLECTION WORKS (COMPLETE STEPS 3.1 through 3.5)</h1>

<h2> In this section, we will implement an AgenticAI Workflow with Reflection. Reflection is the act of pausing to evaluate/judge what was generated before finalizing an answer. In other words, separating generation from judgement so weaknesses can be identified. Specifically, we will implement the following workflow:<br>
* question -> DRAFTER (LLM + web_search) -> draft <br>
* draft -> EVALUATOR (LLM) -> evaluation <br>
* draft & evaluation -> REVISER (LLM) -> revision <br>
NOTE: You may need to iterate on STEPS 3.1-3.5 several times before you are able to settle on the final version for submission.  
</h2>

<h2>
We'll start by specifiying the question we want to pose to the agenticAI workflow. <br>
NOTE: You can test the workflow with different questions, but ensure that your final submission has the original question as stated below.
</h2>

In [9]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

# This is the question we are attempting to answer in Section 3.
# You MUST NOT change this question for your final submission.
# However, feel free to test the drafter->evaluator->reviser agenticAI workflow in Section 3
# with different questions to ensure your code in Steps 3.1-3.5 generalizes well.
question = (
    "What is the latest total investment announced by TSMC for its Arizona fabs, and what does it include?"
)

<h2>[Complete Coding STEP-3.1]: <br>
We'll now define the drafter agent: question -> DRAFTER (LLM + web_search) -> draft. <br>
You will first update the system_prompt (STEP 3.1) as you see fit. You will then execute the cell and examine the draft. </h2>

In [10]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################


# STEP-3.1: You have been provided with a system_prompt for the drafter.
#           Feel free to update the system_prompt if required.
#           But you MUST NOT change the role of the drafter.
draft_system_prompt = (
    f"""
    You are a professional business journalist.
    Use the web_search tool to find the most recent investment figures for TSMC's Arizona fabs.
    Include specific dollar amounts, the number of fabs mentioned, and any government funding details.
    """
)

# We are going to send the question to the drafter
draft_user_prompt = (
    f"""
    Question from user: {question}
    """
)

# Have the LLM model (with web_search) produce a first draft.
draft, query_details = run_openai_query(draft_system_prompt, draft_user_prompt, tools=[openai_web_search])

show("----- DRAFT (WITH WEB_SEARCH) -----", "")
show("Question", draft_user_prompt)
show("Draft", draft)

<h2>[Complete Coding STEP-3.2 and STEP 3.3]: <br>
Now we'll define the evaluator agent: draft -> EVALUATOR (LLM) -> evaluation. <br>
You will first update the system_prompt as you see fit (STEP 3.2) and the user_prompt as indicated (STEP 3.3). You will then execute the cell and examine the evaluation. </h2>

In [11]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

# STEP-3.2: You have been provided with a system_prompt for the evaluator.
#           Feel free to update the system_prompt if required.
#           But you MUST NOT change the role of the evaluator.
evaluation_system_prompt = (
  f"""
  You are a senior editor.
  Critique the draft for factual density, clarity, and source attribution.
  Do not rewrite the text; only provide a list of improvements.
  """
)

# STEP-3.3: You have been provided with a starter user_prompt for the evaluator.
#           It takes the draft from the drafter, and evaluates it.
#           Update the evaluation steps below to provide more concrete guidance to the evaluator.
evaluation_user_prompt = (
    f"""
    Draft from drafter: {draft}
    Evaluation Steps:
    1) Check if the total investment amount (e.g., $65 billion) is clearly stated.
    2) Verify if the production technology (nanometer nodes) is mentioned.
    3) Evaluate if the CHIPS Act grants or loans are included.
    4) Check for professional tone and logical flow.
    5) Identify any missing context regarding the third fab.
    """
)

# Have the LLM model reflect on the draft and provide its evaluation (without web_search).
# We do not provide access to web_search becuase we want the critique to be based on reasoning, not new facts.
evaluation, query_details = run_openai_query(evaluation_system_prompt, evaluation_user_prompt, tools=[])

show("----- EVALUATION (WITHOUT WEB_SEARCH) -----", "")
show("Evaluation", evaluation)

<h2>[Complete Coding STEP-3.4 and STEP 3.5]: <br>
Now we'll define the revisor agent: draft & evaluation -> REVISER (LLM) -> revision. <br>
You will first update the system_prompt as you see fit (STEP 3.4) and the user_prompt as indicated (STEP 3.5). You will then execute the cell and examine the revision. </h2>

In [12]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

# STEP-3.4: You have been provided with a system_prompt for the reviser.
#           Feel free to update the system_prompt if required.
#           But you MUST NOT change the role of the reviser.
revision_system_prompt = (
  f"""
  You are an expert technical writer.
  Your task is to take the original draft and integrate the evaluator's feedback to produce a polished, final report.
  """
)

# STEP-3.5: You have been provided with a starter user_prompt for the reviser.
#           It revises the draft from the drafter based on the evaluation from the evaluator.
#           Update the revision steps below to provide more concrete guidance to the reviser.
revision_user_prompt = (
    f"""
    Draft from drafter: {draft}
    Evaluation from evaluator: {evaluation}
    Revision Steps:
    1) Correct any errors or omissions identified by the evaluator.
    2) Ensure the $65 billion total investment figure is prominent.
    3) Structure the response with clear headings or bullet points for readability.
    4) Ensure all technical specifications (2nm or 3nm processes) are included.
    5) Maintain a neutral, professional business tone.
    """
)

# Have the LLM model revise the draft based on the evaluation (without web_search).
# We do not provide access to web_search becuase we want the revision to integrate, not expand.
revision, query_details = run_openai_query(revision_system_prompt, revision_user_prompt, tools=[])

show("----- Revision (WITHOUT TOOLS) -----", "")
show("Revision", revision)